<center>
    <h1>2.2.0 Classify the Analysis Corpus</h1>
<hr style="width: 70%; margin: 1.5em auto; border: none; border-top: 1.5px solid #FFF;">
    <h2>SciBERT classifier trained on the annotation-support corpus</h2>
</center>


<center>
    <h2>Configuration, paths, and classifier setup</h2>
<hr style="width: 70%; margin: 1.5em auto; border: none; border-top: 1.5px solid #FFF;">
</center>

Load the trained SciBERT classifier, resolve the main analysis-corpus paths, and prepare the inference helpers used to label the full corpus.


In [ ]:
from __future__ import annotations

import os
import shutil
import sys
from pathlib import Path


def _find_workspace_root() -> Path:
    env_root = os.environ.get("WORKSPACE_ROOT", "").strip()
    if env_root:
        candidate = Path(env_root).expanduser().resolve()
        if (candidate / "config" / "workspace.json").exists():
            return candidate
    for start in [Path.cwd(), *Path.cwd().parents]:
        if (start / "config" / "workspace.json").exists():
            return start
    raise FileNotFoundError(
        "Could not locate workspace root from WORKSPACE_ROOT or the current working directory."
    )


WORKSPACE_DIR = _find_workspace_root()
PAPER_ROOT = WORKSPACE_DIR.parent
RESEARCH_ROOT = PAPER_ROOT.parent
SHARED_ASSETS_DIR = RESEARCH_ROOT / "shared-assets"
SHARED_CODE_DIR = SHARED_ASSETS_DIR / "code"
if str(SHARED_CODE_DIR) not in sys.path:
    sys.path.insert(0, str(SHARED_CODE_DIR))

CODE_DIR = WORKSPACE_DIR / "code"
NOTEBOOKS_DIR = CODE_DIR / "notebooks"
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_DIR))

from workspace_rooting.workspace_paths import canonical_workspace_paths

paths = canonical_workspace_paths(WORKSPACE_DIR)
WORKSPACE_DIR = paths["workspace"]
CODE_DIR = paths["code"]
CONFIG_DIR = paths["config"]
DATA_DIR = paths["data"]
OUTPUTS_DIR = paths["outputs"]
DOCS_DIR = paths["docs"]
LOCAL_DIR = paths["local"]
SHARED_ASSETS_DIR = paths["shared_assets"]
NOTEBOOKS_DIR = CODE_DIR / "notebooks"

print("workspace_root:", WORKSPACE_DIR)
print("shared_assets_root:", SHARED_ASSETS_DIR)

# Repo mode omits manuscript and Overleaf export surfaces.
def record_figure_output(source: Path, target_name: str | None = None) -> Path:
    return Path(source)

def record_table_output(source: Path, target_name: str | None = None) -> Path:
    return Path(source)

import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib_venn import venn3
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy.ndimage import uniform_filter1d
from scipy.stats import gaussian_kde
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer

from notebook_theme import activate_theme, colors, style_axis

theme_name = "paper"
theme = activate_theme(theme_name)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

RUN_ID = "analysis_corpus"
CLASSIFIER_RUN_ID = "scibert_classifier_3labs"
FORCE_RECLASSIFY_ALL = True

CORPUS_BINS_DIR = LOCAL_DIR / "corpora" / RUN_ID / "bins"
RUN_ROOT = OUTPUTS_DIR / "analytical-results" / RUN_ID
CORPUS_DATA_DIR = RUN_ROOT / "corpus" / "data"
CORPUS_PLOTS_DIR = RUN_ROOT / "corpus" / "plots" / theme_name
CLASSIFIED_CORPUS_PATH = RUN_ROOT / "corpus" / "corpus_classified.jsonl"
SCIBERT_BASE = LOCAL_DIR / "models" / "scibert_scivocab_uncased"
BEST_MODEL_PATH = OUTPUTS_DIR / "analytical-results" / "models" / "scibert" / f"{CLASSIFIER_RUN_ID}_best.pt"
THRESHOLDS_PATH = OUTPUTS_DIR / "analytical-results" / "models" / "scibert" / f"{CLASSIFIER_RUN_ID}_thresholds.json"

for path in [CORPUS_DATA_DIR, CORPUS_PLOTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
LABELS = ["A", "B", "C"]

print("CORPUS_BINS_DIR:", CORPUS_BINS_DIR, "exists:", CORPUS_BINS_DIR.exists())
print("CLASSIFIED_CORPUS_PATH:", CLASSIFIED_CORPUS_PATH, "exists:", CLASSIFIED_CORPUS_PATH.exists())
print("BEST_MODEL_PATH:", BEST_MODEL_PATH, "exists:", BEST_MODEL_PATH.exists())
print("THRESHOLDS_PATH:", THRESHOLDS_PATH, "exists:", THRESHOLDS_PATH.exists())
print("FORCE_RECLASSIFY_ALL:", FORCE_RECLASSIFY_ALL)
print("Using device:", DEVICE)


<center>
    <h2>Helper, functions, and definitions</h2>
<hr style="width: 70%; margin: 1.5em auto; border: none; border-top: 1.5px solid #FFF;">
</center>


In [ ]:
with open(THRESHOLDS_PATH, "r", encoding="utf-8") as f:
    threshold_info = json.load(f)
LABEL_THRESHOLDS = {
    label: float(threshold_info["per_label"][label]["threshold"])
    for label in LABELS
}
print("Using corpus informed per-label thresholds:", LABEL_THRESHOLDS)


def load_jsonl(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]


def row_signature(row) -> tuple:
    sent = " ".join(str(row["sent"]).split())
    return (
        str(row.get("bibcode") or ""),
        int(row["year"]),
        sent,
        int(row["start"]),
        int(row["end"]),
        bool(row.get("has_math", False)),
    )


def grouped_existing_rows(path: Path) -> dict[str, list[dict]]:
    if not path.exists():
        return {}
    grouped = {}
    for row in load_jsonl(path):
        grouped.setdefault(str(row["bin"]), []).append(row)
    return grouped


def cache_bin_matches(corpus_records: list[dict], classified_records: list[dict]) -> bool:
    if len(corpus_records) != len(classified_records):
        return False
    corpus_sig = [row_signature(row) for row in corpus_records]
    cached_sig = [row_signature(row) for row in classified_records]
    return corpus_sig == cached_sig


class SciBERTClassifier(nn.Module):
    def __init__(self, base_model, n_labels=3, dropout=0.1):
        super().__init__()
        self.bert = base_model
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.bert.config.hidden_size, n_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls = outputs.last_hidden_state[:, 0, :]
        return self.classifier(self.dropout(cls))


class MentionDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len=256):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding=True,
            max_length=max_len,
            return_tensors="pt",
        )

    def __len__(self):
        return self.encodings["input_ids"].shape[0]

    def __getitem__(self, idx):
        return {
            "input_ids": self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
        }


def assign_labels(prob_A, prob_B, prob_C):
    probs = {"A": prob_A, "B": prob_B, "C": prob_C}
    return [label for label in LABELS if probs[label] >= LABEL_THRESHOLDS[label]]


def sample_category(cat, bin_label, n=10, threshold=0.8):
    subset = df[
        (df["bin"] == bin_label)
        & (df[f"prob_{cat}"] >= threshold)
        & (df["labels"].apply(lambda xs: cat in xs))
    ]

    print(f"\n=== Category {cat} | {bin_label} | prob >= {threshold} ===")
    if subset.empty:
        print("No rows.")
        return

    sample = subset.sample(min(n, len(subset)), random_state=42)
    for _, row in sample.iterrows():
        print(f"  [{row['year']}] {row['sent']}")
        print(f"  probs: A={row['prob_A']:.2f} B={row['prob_B']:.2f} C={row['prob_C']:.2f}\n")


def sample_label_combo(label_combo, bin_label, n=10, seed=42):
    target = tuple(sorted(label_combo))
    subset = df[(df["bin"] == bin_label) & (df["labels"].apply(lambda xs: tuple(sorted(xs)) == target))]

    pretty = "+".join(target) if target else "none"
    print(f"\n=== Label combo {pretty} | {bin_label} ===")
    if subset.empty:
        print("No rows.")
        return

    sample = subset.sample(min(n, len(subset)), random_state=seed)
    for _, row in sample.iterrows():
        print(f"  [{row['year']}] {row['sent']}")
        print(f"  labels={row['labels']}")
        print(f"  probs: A={row['prob_A']:.2f} B={row['prob_B']:.2f} C={row['prob_C']:.2f}\n")


tokenizer = AutoTokenizer.from_pretrained(SCIBERT_BASE, use_fast=True)
model = SciBERTClassifier(AutoModel.from_pretrained(SCIBERT_BASE).to(DEVICE), n_labels=3).to(DEVICE)
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=DEVICE, weights_only=True))
model.eval()
print(f"Classifier loaded on {DEVICE}")


<center>
    <h1>Run corpus classification and write the labeled analysis corpus</h1>
<hr style="width: 70%; margin: 1.5em auto; border: none; border-top: 1.5px solid #FFF;">
</center>


In [ ]:
existing_by_bin = grouped_existing_rows(CLASSIFIED_CORPUS_PATH)
all_results = []
reused_bins = []
recomputed_bins = []

for bin_file in sorted(CORPUS_BINS_DIR.glob("*.jsonl"), key=lambda path: int(path.stem.split("-")[0])):
    bin_label = bin_file.stem
    records = load_jsonl(bin_file)
    cached_rows = existing_by_bin.get(bin_label, [])

    if (not FORCE_RECLASSIFY_ALL) and cached_rows and cache_bin_matches(records, cached_rows):
        all_results.extend(cached_rows)
        reused_bins.append(bin_label)
        print(f"Reusing cached classifications for {bin_label}: {len(cached_rows):,} rows")
        continue

    dataset = MentionDataset([row["sent"] for row in records], tokenizer)
    loader = DataLoader(dataset, batch_size=128, shuffle=False, num_workers=0, pin_memory=False)

    all_probs = []
    with torch.no_grad():
        for batch in tqdm(loader, desc=bin_label):
            logits = model(batch["input_ids"].to(DEVICE), batch["attention_mask"].to(DEVICE))
            all_probs.append(torch.sigmoid(logits).cpu().numpy())

    bin_results = []
    for i, (row, prob) in enumerate(zip(records, np.vstack(all_probs))):
        prob_A, prob_B, prob_C = map(float, prob)
        bin_results.append({
            "id": f"{bin_label}_{i}",
            "bin": bin_label,
            "year": int(row["year"]),
            "bibcode": row.get("bibcode"),
            "sent": row["sent"],
            "start": int(row["start"]),
            "end": int(row["end"]),
            "has_math": bool(row.get("has_math", False)),
            "prob_A": prob_A,
            "prob_B": prob_B,
            "prob_C": prob_C,
            "labels": assign_labels(prob_A, prob_B, prob_C),
        })

    all_results.extend(bin_results)
    recomputed_bins.append(bin_label)
    if cached_rows:
        print(f"Recomputed {bin_label}: cache drifted ({len(cached_rows):,} -> {len(bin_results):,} rows)")
    else:
        print(f"Classified new bin {bin_label}: {len(bin_results):,} rows")

all_results = sorted(all_results, key=lambda row: (int(str(row["bin"]).split("-")[0]), int(row["id"].split("_")[-1])))

with open(CLASSIFIED_CORPUS_PATH, "w", encoding="utf-8") as f:
    for row in all_results:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print(f"Classified {len(all_results):,} sentences total")
print(f"Reused bins: {reused_bins}")
print(f"Recomputed bins: {recomputed_bins}")
print(f"Saved to {CLASSIFIED_CORPUS_PATH}")


<center>
    <h2>Summarize label assignments by bin and write the core metadata tables</h2>
<hr style="width: 70%; margin: 1.5em auto; border: none; border-top: 1.5px solid #FFF;">
</center>


In [ ]:
df_cls = pd.DataFrame(all_results)
df_cls["label_str"] = df_cls["labels"].apply(lambda x: "+".join(sorted(x)) if x else "none")
print(df_cls["label_str"].value_counts().head(10))

label_mix = pd.crosstab(
    index=df_cls["bin"],
    columns=df_cls["label_str"],
    normalize="index",
).round(3)

print("\n", label_mix, "\n")
label_mix_path = CORPUS_DATA_DIR / "label_mix_by_bin.csv"
label_mix.to_csv(label_mix_path, index=True)
print(f"Saved label mix -> {label_mix_path}\n")
display(label_mix)


<center>
    <h2>Plot corpus-level diagnostics and sample sentences for auditing</h2>
<hr style="width: 70%; margin: 1.5em auto; border: none; border-top: 1.5px solid #FFF;">
</center>


In [ ]:
records = load_jsonl(CLASSIFIED_CORPUS_PATH)
df = pd.DataFrame(records)

for cat in LABELS:
    df[f"has_{cat}"] = df["labels"].apply(lambda x: cat in x)

df["dominant"] = df[["prob_A", "prob_B", "prob_C"]].idxmax(axis=1).map({
    "prob_A": "A",
    "prob_B": "B",
    "prob_C": "C",
})
df["dominant"] = df.apply(
    lambda r: r["dominant"] if (r["has_A"] or r["has_B"] or r["has_C"]) else "none",
    axis=1,
)

df["n_labels"] = df["labels"].apply(len)
bin_order = sorted(df["bin"].unique(), key=lambda s: int(s.split("-")[0]))
props = df.groupby("bin")[[f"has_{c}" for c in LABELS]].mean().loc[bin_order]
props.columns = LABELS

print(props.round(3))

props.plot(
    kind="bar",
    figsize=(12, 5),
    title="Category proportions per bin",
    color=[colors["A"], colors["B"], colors["C"]],
)

plt.ylabel("Proportion of sentences")
plt.xlabel("")
plt.xticks(rotation=45, ha="right")
plt.legend(loc="upper right", ncols=3, frameon=False)
plt.tight_layout()
plt.savefig(CORPUS_PLOTS_DIR / "category_proportions.png", dpi=300, bbox_inches="tight")
plt.savefig(CORPUS_PLOTS_DIR / "category_proportions.pdf", bbox_inches="tight")
record_figure_output(CORPUS_PLOTS_DIR / "category_proportions.png")
record_figure_output(CORPUS_PLOTS_DIR / "category_proportions.pdf")
print(f"Saved PNG and PDF figure -> {CORPUS_PLOTS_DIR}")
plt.show()


In [ ]:
# ── Meta-data ────────────────────────────────────────────────────
cat_meta = globals().get("cat_meta", {
    "A": {"label": "A"},
    "B": {"label": "B"},
    "C": {"label": "C"},
})

# normalize labels into sets
label_sets = df["labels"].apply(lambda x: set(x) if isinstance(x, (list, tuple, set)) else set())

only_A = (label_sets == {"A"}).sum()
only_B = (label_sets == {"B"}).sum()
only_C = (label_sets == {"C"}).sum()
AB = (label_sets == {"A", "B"}).sum()
AC = (label_sets == {"A", "C"}).sum()
BC = (label_sets == {"B", "C"}).sum()
ABC = (label_sets == {"A", "B", "C"}).sum()

fig, ax = plt.subplots(figsize=(7, 7))

# ── Venn-sets ────────────────────────────────────────────────────
# Order: (A, B, AB, C, AC, BC, ABC)
v = venn3(
    subsets=(only_A, only_B, AB, only_C, AC, BC, ABC),
    set_labels=(
        cat_meta["A"]["label"],
        cat_meta["B"]["label"],
        cat_meta["C"]["label"],
    ),
    set_colors=(
        colors["A"],
        colors["B"],
        colors["C"],
    ),
    alpha=0.6,
    ax=ax,
)

if v.set_labels:
    for label in v.set_labels:
        if label is not None:
            label.set_fontsize(11)

if v.subset_labels:
    for label in v.subset_labels:
        if label is not None:
            label.set_fontsize(9)

ax.set_title(
    "Venn diagram of semantic label overlap per sentence",
    fontsize=13,
)

plt.tight_layout()
plt.savefig(CORPUS_PLOTS_DIR / "venn_diagram_labels.png", dpi=300, bbox_inches="tight")
plt.savefig(CORPUS_PLOTS_DIR / "venn_diagram_labels.pdf", bbox_inches="tight")
record_figure_output(CORPUS_PLOTS_DIR / "venn_diagram_labels.png")
record_figure_output(CORPUS_PLOTS_DIR / "venn_diagram_labels.pdf")
print(f"Saved PNG and PDF figure -> {CORPUS_PLOTS_DIR}")
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharex=True)

cat_meta = {
    "A": {"label": "A | Referential", "color": colors["A"], "has": "has_A", "prob": "prob_A"},
    "B": {"label": "B | Role",        "color": colors["B"], "has": "has_B", "prob": "prob_B"},
    "C": {"label": "C | Model-param", "color": colors["C"], "has": "has_C", "prob": "prob_C"},
}

yearly = df.groupby("year").agg(
    n_A=("has_A", "sum"),
    n_B=("has_B", "sum"),
    n_C=("has_C", "sum"),
    mean_prob_A=("prob_A", "mean"),
    mean_prob_B=("prob_B", "mean"),
    mean_prob_C=("prob_C", "mean"),
    std_prob_A=("prob_A", "std"),
    std_prob_B=("prob_B", "std"),
    std_prob_C=("prob_C", "std"),
    total=("has_A", "count"),
).reset_index()

yearly["ratio_A"] = yearly["n_A"] / yearly["total"]
yearly["ratio_B"] = yearly["n_B"] / yearly["total"]
yearly["ratio_C"] = yearly["n_C"] / yearly["total"]

for cat in "ABC":
    yearly[f"conf_{cat}"] = (
        df[df[f"has_{cat}"]]
        .groupby("year")[f"prob_{cat}"]
        .mean()
        .reindex(yearly["year"])
        .values
    )

ax_A = axes[0, 0]
ax_B = axes[0, 1]
ax_C = axes[1, 0]
ax_std = axes[1, 1]

for ax in [ax_A, ax_B, ax_C, ax_std]:
    style_axis(ax, theme)


for ax, cat in [(ax_A, "A"), (ax_B, "B"), (ax_C, "C")]:
    color_pri = colors[cat]
    color_sec = colors[f"{cat}_secondary"]

    prop = pd.Series(yearly[f"ratio_{cat}"]).ffill()
    conf = pd.Series(yearly[f"conf_{cat}"]).ffill()

    prop_sm = uniform_filter1d(prop, size=3)
    conf_sm = uniform_filter1d(conf, size=3)

    ax.fill_between(
        yearly["year"],
        prop_sm,
        conf_sm,
        alpha=0.15,
        color=color_pri,
    )
    ax.plot(
        yearly["year"],
        yearly[f"ratio_{cat}"],
        color=color_pri,
        linewidth=0.5,
        alpha=0.6,
    )
    ax.plot(
        yearly["year"],
        prop_sm,
        color=color_pri,
        linewidth=1.5,
        label="Proportion (hard label)",
    )
    ax.plot(
        yearly["year"],
        yearly[f"conf_{cat}"],
        color=color_sec,
        linewidth=0.8,
        alpha=0.6,
    )
    ax.plot(
        yearly["year"],
        conf_sm,
        color=color_sec,
        linewidth=1.5,
        linestyle="--",
        label="Mean prob (positive)",
    )

    ax.set_ylim(0, 1)
    ax.set_title(cat_meta[cat]["label"], fontsize=10)
    ax.grid(alpha=0.2)

    leg = ax.legend(fontsize=7, loc="upper left")
    frame = leg.get_frame()
    frame.set_boxstyle("square,pad=0.2")
    frame.set_linewidth(0.8)
    frame.set_alpha(1.0)
    frame.set_facecolor(theme["bg"])
    frame.set_edgecolor(theme["fg"])

for cat in "ABC":
    color_pri = colors[cat]
    std = pd.Series(yearly[f"std_prob_{cat}"]).ffill()
    std_sm = uniform_filter1d(std, size=3)

    ax_std.plot(
        yearly["year"],
        yearly[f"std_prob_{cat}"],
        color=color_pri,
        linewidth=0.8,
        alpha=0.6,
    )
    ax_std.plot(
        yearly["year"],
        std_sm,
        color=color_pri,
        linewidth=1.5,
        label=cat_meta[cat]["label"],
    )

ax_std.set_ylim(0, 0.5)
ax_std.set_title("Classifier uncertainty (std dev)", fontsize=10)
ax_std.set_xlabel("Year", fontsize=10)
ax_std.set_ylabel("Std dev", fontsize=10)
ax_std.grid(alpha=0.2)

dev_leg = ax_std.legend(fontsize=9, loc="upper left")
frame = dev_leg.get_frame()
frame.set_boxstyle("square,pad=0.2")
frame.set_linewidth(0.8)
frame.set_alpha(1.0)
frame.set_facecolor(theme["bg"])
frame.set_edgecolor(theme["fg"])

for ax in [ax_A, ax_B]:
    ax.tick_params(labelbottom=False)
for ax in [ax_A, ax_C]:
    ax.set_ylabel("Proportion / Mean prob", fontsize=10)
ax_C.set_xlabel("Year", fontsize=10)

fig.suptitle(
    "Prevalence of semantic categories\n"
    "Confidence and uncertainty by year\n"
    "Dark matter corpus (SciBERT)",
    fontsize=13,
    y=0.98,
)

plt.tight_layout()
plt.savefig(CORPUS_PLOTS_DIR / "category_trends_2x2.png", dpi=300, bbox_inches="tight")
plt.savefig(CORPUS_PLOTS_DIR / "category_trends_2x2.pdf", bbox_inches="tight")
record_figure_output(CORPUS_PLOTS_DIR / "category_trends_2x2.png")
record_figure_output(CORPUS_PLOTS_DIR / "category_trends_2x2.pdf")
print(f"Saved PNG and PDF figure -> {CORPUS_PLOTS_DIR}")

plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
ax  = axes[0]
ax2 = axes[1]
for a in [ax, ax2]:
    style_axis(a, theme)

# ── 1. Percentage stacked area ───────────────────────────────────────────────
df["dominant"] = df[["prob_A", "prob_B", "prob_C"]].idxmax(axis=1).map({
    "prob_A": "A", "prob_B": "B", "prob_C": "C"
})

# Unlabeled sentences override dominant — no label fired above threshold
df["dominant"] = df.apply(
    lambda r: r["dominant"] if (r["has_A"] or r["has_B"] or r["has_C"]) else "none",
    axis=1
)

# Now every sentence belongs to exactly one of: A, B, C, none
print(df["dominant"].value_counts())

yearly_stack = df.groupby(["year", "dominant"]).size().unstack(fill_value=0).reset_index()

# Ensure all columns exist
for col in ["A", "B", "C", "none"]:
    if col not in yearly_stack.columns:
        yearly_stack[col] = 0

yearly_stack["total"] = yearly_stack[["A", "B", "C", "none"]].sum(axis=1)
for col in ["A", "B", "C", "none"]:
    yearly_stack[f"pct_{col}"] = yearly_stack[col] / yearly_stack["total"] * 100
for col in ["pct_A", "pct_B", "pct_C", "pct_none"]:
    yearly_stack[col] = yearly_stack[col].rolling(3, center=True, min_periods=1).mean()
yr = yearly_stack["year"]

ax.fill_between(yr, 0,
    yearly_stack["pct_none"],
    color=colors["none"], alpha=0.85, label="Unlabeled")

ax.fill_between(yr,
    yearly_stack["pct_none"],
    yearly_stack["pct_none"] + yearly_stack["pct_C"],
    color=colors["C"], alpha=0.85, label=cat_meta["C"]["label"])

ax.fill_between(yr,
    yearly_stack["pct_none"] + yearly_stack["pct_C"],
    yearly_stack["pct_none"] + yearly_stack["pct_C"] + yearly_stack["pct_B"],
    color=colors["B"], alpha=0.85, label=cat_meta["B"]["label"])

ax.fill_between(yr,
    yearly_stack["pct_none"] + yearly_stack["pct_C"] + yearly_stack["pct_B"],
    100, color=colors["A"], alpha=0.85, label=cat_meta["A"]["label"])

ax.set_ylim(0, 100)
ax.set_xlim(yr.min(), yr.max())
ax.set_ylabel("% of sentences", fontsize=9)
ax.set_xlabel("Year", fontsize=9)
ax.set_title("Label composition by year\n(% stacked)", fontsize=12)
ax.legend(loc="upper left", fontsize=8)
ax.grid(alpha=0.2, axis="y")

# ── 2. Probability distribution of labeled sentences over time ───────────────
df_labeled = df[df["has_A"] | df["has_B"] | df["has_C"]].copy()
df_labeled["max_prob"] = df_labeled[["prob_A", "prob_B", "prob_C"]].max(axis=1)
df_labeled["period"] = pd.cut(df_labeled["year"],
                               bins=[1985, 1995, 2005, 2015, 2025],
                               labels=["1986–1995", "1996–2005", "2006–2015", "2016–2025"])

# 1. period_order should come from df_labeled, not df
period_order = ["1986–1995", "1996–2005", "2006–2015", "2016–2025"]  # explicit order
n_periods = len(period_order)

# 2. cmap colors based on n_periods, not n_bins
cmap = plt.get_cmap("Spectral") #inferno #flare
period_colors = [cmap(0.2 + 0.7 * i / (n_periods - 1)) for i in range(n_periods)]

for i, period_label in enumerate(period_order):
    vals = df_labeled[df_labeled["period"] == period_label]["max_prob"].dropna().values
    if len(vals) < 10:
        continue

    kde    = gaussian_kde(vals, bw_method=0.08)
    x_grid = np.linspace(0.4, 0.95, 300)
    y_kde  = kde(x_grid)
    y_plot = y_kde / y_kde.max()   # peak-normalize

    ax2.plot(x_grid, y_plot, color=period_colors[i],
             linewidth=1.5, label=period_label, alpha=0.9)
    ax2.fill_between(x_grid, 0, y_plot,   # subtle fill for readability
                     color=period_colors[i], alpha=0.08)

ax2.set_xlim(0.4, 0.95)
x_grid = np.linspace(0.85, 1.0, 300)
ax2.set_xlabel("Max assigned probability", fontsize=9)
ax2.set_ylim(0, 1.05)
ax2.set_ylabel("Normalized density", fontsize=9)
ax2.set_title("Distribution of classifier probabilities\nfor labeled sentences", fontsize=12)
ax2.legend(fontsize=9, title_fontsize=10,
           ncol=1, loc="upper left")
ax2.grid(alpha=0.2)

fig.suptitle("Class proportions [collapsed to single class by highest probability\n"
             "and classifier confidence dark matter corpus (SciBERT)",
             fontsize=14, y=0.95)
plt.tight_layout()
plt.savefig(CORPUS_PLOTS_DIR / "label_composition.png", dpi=300, bbox_inches="tight")
plt.savefig(CORPUS_PLOTS_DIR / "label_composition.pdf", bbox_inches="tight")
record_figure_output(CORPUS_PLOTS_DIR / "label_composition.png")
record_figure_output(CORPUS_PLOTS_DIR / "label_composition.pdf")
print(f"Saved PNG and PDF -> {CORPUS_PLOTS_DIR}")
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
ax1, ax2, ax3, ax4 = axes.flatten()

for ax in axes.flatten():
    style_axis(ax, theme)

# 1. Overall probability distributions
for cat, color in zip("ABC", [colors["A"], colors["B"], colors["C"]]):
    vals = df_cls[f"prob_{cat}"].dropna().values
    ax1.hist(vals, bins=40, density=True, alpha=0.35, color=color, label=cat, edgecolor="none")

ax1.set_title("Overall probability distributions")
ax1.set_xlabel("Predicted probability")
ax1.set_ylabel("Density")
ax1.legend(frameon=False)

# 2. prob_C split by whether C fired
vals_c_yes = df[df["labels"].apply(lambda xs: "C" in xs)]["prob_C"].dropna().values
vals_c_no  = df[~df["labels"].apply(lambda xs: "C" in xs)]["prob_C"].dropna().values

ax2.hist(vals_c_no, bins=40, density=True, alpha=0.45, color="gray", label="C not assigned", edgecolor="none")
ax2.hist(vals_c_yes, bins=40, density=True, alpha=0.45, color=colors["C"], label="C assigned", edgecolor="none")
ax2.axvline(LABEL_THRESHOLDS["C"], color=theme["fg"], linestyle="--", linewidth=1)
ax2.set_title("prob_C by assignment")
ax2.set_xlabel("prob_C")
ax2.set_ylabel("Density")
ax2.legend(frameon=False)

# 3. prob_C by dominant label
for dom, color in zip(["A", "B", "C"], [colors["A"], colors["B"], colors["C"]]):
    vals = df[df["dominant"] == dom]["prob_C"].dropna().values
    if len(vals):
        ax3.hist(vals, bins=40, density=True, alpha=0.30, color=color, label=f"dominant {dom}", edgecolor="none")

ax3.axvline(LABEL_THRESHOLDS["C"], color=theme["fg"], linestyle="--", linewidth=1)
ax3.set_title("prob_C by dominant label")
ax3.set_xlabel("prob_C")
ax3.set_ylabel("Density")
ax3.legend(frameon=False)

# 4. Boxplot for quick comparison
box_data = [
    df["prob_A"].dropna().values,
    df["prob_B"].dropna().values,
    df["prob_C"].dropna().values,
]
bp = ax4.boxplot(
    box_data,
    tick_labels=["A", "B", "C"],
    patch_artist=True,
    widths=0.6,
)

for patch, color in zip(bp["boxes"], [colors["A"], colors["B"], colors["C"]]):
    patch.set_facecolor(color)
    patch.set_alpha(0.5)
    patch.set_edgecolor(theme["fg"])

for element in ["whiskers", "caps", "medians"]:
    for item in bp[element]:
        item.set_color(theme["fg"])

ax4.set_title("Probability spread by label")
ax4.set_ylabel("Predicted probability")

fig.suptitle("SciBERT probability diagnostics", fontsize=14, y=0.98)
plt.tight_layout()
plt.savefig(CORPUS_PLOTS_DIR / "scibert_probability_diagnostics.png", dpi=300, bbox_inches="tight")
record_figure_output(CORPUS_PLOTS_DIR / "scibert_probability_diagnostics.png")
plt.show()


## Inspect high-confidence category examples and overlap cases


In [ ]:
for cat in "ABC":
    sample_category(cat, "1991-1995")
    sample_category(cat, "2001-2005")
    sample_category(cat, "2021-2025")


print("=== Full corpus label summary ===")
print(f"Total sentences: {len(df)}")
print(f"\nPer-category counts and proportions:")
for cat in "ABC":
    n = df[f"has_{cat}"].sum()
    pct = 100 * n / len(df)
    print(f"  {cat}: {n:6d} ({pct:.1f}%)")

print(f"\nMulti-label sentences:")
df["n_labels"] = df["labels"].apply(len)
print(df["n_labels"].value_counts().sort_index())

print(f"\nUnlabeled (no category assigned):")
unlabeled = df["labels"].apply(lambda x: len(x) == 0).sum()
print(f"  {unlabeled} ({100*unlabeled/len(df):.1f}%)")


In [ ]:
# Audit the overlap cases we care most about
for bin_label in ["1991-1995", "2001-2005", "2021-2025"]:
    sample_label_combo(["A", "C"], bin_label, n=10, seed=42)
    sample_label_combo(["B", "C"], bin_label, n=10, seed=43)
    sample_label_combo(["C"], bin_label, n=10, seed=44)
